# World Events MCP Server — News Intelligence Colab Notebook

This notebook installs and tests the **World Events MCP Server** — a real-time
news intelligence platform powered by free public APIs (no API keys required).

**Tools available:**
- `intel_news_feed` — RSS aggregation across 24 categories / 119 feeds
- `intel_trending_keywords` — Keyword spike detection from recent headlines
- `intel_gdelt_search` — GDELT 2.0 Doc API: 65-language global news search
- `intel_status` — Server health and cache statistics

## 1. Install

In [18]:
# Clone the repo and install the package
!git clone https://github.com/dshipley71/WorldEventsPipeline.git 2>/dev/null || echo 'Already cloned'
!pip install -q -e '/content/WorldEventsPipeline/world-events-mcp'

Already cloned
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for world-events-mcp (pyproject.toml) ... done


## 2. MCP Client Helper

The server speaks **JSON-RPC 2.0 over stdio**. This helper manages the subprocess and handles the protocol handshake.

In [19]:
import json
import subprocess
import threading
import queue
import time
import sys
from IPython.display import display, JSON


class MCPClient:
    """Minimal synchronous MCP client over stdio."""

    def __init__(self):
        self._proc = None
        self._reader_thread = None
        self._responses: queue.Queue = queue.Queue()
        self._stderr_lines: list[str] = []
        self._id = 0

    # ------------------------------------------------------------------
    # Lifecycle
    # ------------------------------------------------------------------

    def connect(self):
        """Start the MCP server subprocess and run the initialize handshake."""
        self._proc = subprocess.Popen(
            ["world-events-mcp"],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            bufsize=1,
        )
        # Background thread: read stdout lines and push to response queue
        self._reader_thread = threading.Thread(
            target=self._read_loop, daemon=True
        )
        self._reader_thread.start()

        # MCP initialize handshake
        resp = self._send("initialize", {
            "protocolVersion": "2024-11-05",
            "capabilities": {},
            "clientInfo": {"name": "colab-tester", "version": "1.0"},
        })
        # Acknowledge initialization
        self._notify("notifications/initialized", {})
        print(f"✅ Connected — server: {resp.get('result', {}).get('serverInfo', {})}")
        return self

    def disconnect(self):
        """Terminate the server process."""
        if self._proc:
            self._proc.terminate()
            self._proc = None
        print("Disconnected.")

    def __enter__(self):
        return self.connect()

    def __exit__(self, *_):
        self.disconnect()

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------

    def list_tools(self) -> list[dict]:
        """Return all available tool definitions."""
        resp = self._send("tools/list", {})
        return resp.get("result", {}).get("tools", [])

    def call_tool(self, name: str, arguments: dict | None = None, timeout: float = 30.0) -> dict:
        """Invoke a named tool with the given arguments."""
        resp = self._send(
            "tools/call",
            {"name": name, "arguments": arguments or {}},
            timeout=timeout,
        )
        result = resp.get("result", {})
        # Unwrap MCP content envelope
        contents = result.get("content", [])
        for block in contents:
            if block.get("type") == "text":
                try:
                    return json.loads(block["text"])
                except json.JSONDecodeError:
                    return {"raw": block["text"]}
        return result

    # ------------------------------------------------------------------
    # Internal
    # ------------------------------------------------------------------

    def _next_id(self) -> int:
        self._id += 1
        return self._id

    def _write(self, msg: dict):
        line = json.dumps(msg) + "\n"
        self._proc.stdin.write(line)
        self._proc.stdin.flush()

    def _send(self, method: str, params: dict, timeout: float = 30.0) -> dict:
        msg_id = self._next_id()
        self._write({"jsonrpc": "2.0", "id": msg_id, "method": method, "params": params})
        deadline = time.time() + timeout
        while time.time() < deadline:
            try:
                resp = self._responses.get(timeout=1.0)
                if resp.get("id") == msg_id:
                    return resp
                # put back if it's for another id
                self._responses.put(resp)
            except queue.Empty:
                pass
        raise TimeoutError(f"No response for method={method!r} within {timeout}s")

    def _notify(self, method: str, params: dict):
        """Send a notification (no id, no response expected)."""
        self._write({"jsonrpc": "2.0", "method": method, "params": params})

    def _read_loop(self):
        for line in self._proc.stdout:
            line = line.strip()
            if not line:
                continue
            try:
                msg = json.loads(line)
                if "id" in msg:
                    self._responses.put(msg)
                # drop server-sent notifications
            except json.JSONDecodeError:
                pass


# Instantiate and connect once for the whole notebook
client = MCPClient()
client.connect()

✅ Connected — server: {'name': 'world-events-mcp', 'version': '1.26.0'}


## 3. List Available Tools

In [20]:
tools = client.list_tools()
print(f"Total tools: {len(tools)}\n")
for t in tools:
    print(f"  {t['name']:<40} {t.get('description', '')[:70]}")

Total tools: 70

  intel_macro_signals                      Get 7 key macro signals: Fear & Greed, mempool fees, DXY, VIX, gold, 1
  intel_world_bank_indicators              Get World Bank development indicators (GDP, inflation, unemployment) f
  intel_earthquakes                        Get recent earthquakes from USGS. No API key needed.
  intel_humanitarian_summary               Get humanitarian crisis datasets from HDX. No API key needed. Optional
  intel_military_flights                   Track military aircraft via OpenSky Network (ICAO hex + callsign filte
  intel_theater_posture                    Get military aircraft presence across 5 theaters: European, Indo-Pacif
  intel_aircraft_details                   Get aircraft details from hexdb.io by ICAO24 hex code (free, no API ke
  intel_internet_outages                   Get internet outages from IODA (Georgia Tech Internet Intelligence) — 
  intel_cable_health                       Assess undersea cable corridor health from NGA

## 4. News — Feed with Category Filters

In [21]:
result = client.call_tool("intel_news_feed", {"categories": ["geopolitics", "military"], "limit": 15})

articles = result.get("articles", [])
print(f"Latest {len(articles)} articles:\n")
for a in articles:
    print(f"  [{a.get('category',''):<12}] {a.get('title','')[:80]}")
    print(f"    {a.get('published','')[:19]}  {a.get('source','')}")
    print()

Latest 0 articles:



## 5. Server Health — Data Source Status

In [22]:
result = client.call_tool("intel_status")

cache = result.get("cache", {})
cb    = result.get("circuit_breakers", {})

print(f"Cache entries: {cache.get('entries', '?')}  |  "
      f"Cache size: {cache.get('size_mb', '?')} MB\n")

open_breakers = [k for k, v in cb.items() if v.get("state") == "open"]
if open_breakers:
    print(f"⚠️  Open circuit breakers: {', '.join(open_breakers)}\n")
else:
    print("✅ All circuit breakers closed\n")

print("Data sources:")
for category, feeds in result.get("sources", {}).items():
    print(f"  {category}: {feeds}")

Cache entries: ?  |  Cache size: ? MB

✅ All circuit breakers closed

Data sources:
  markets: ['yahoo-finance', 'coingecko', 'alternative-me', 'mempool']
  economic: ['world-bank']
  natural: ['usgs']
  conflict: ['ucdp', 'hdx']
  military: ['hexdb', 'adsblol']
  infrastructure: ['ioda', 'nga-msi']
  maritime: ['nga-msi']
  climate: ['open-meteo']
  news: ['rss-aggregator', 'gdelt']
  intelligence: ['ollama', 'world-bank', 'hdx', 'usgs']
  prediction: ['polymarket']
  displacement: ['unhcr']
  aviation: ['faa']
  cyber: ['feodo-tracker', 'cisa-kev', 'sans-dshield', 'urlhaus']
  space_weather: ['noaa-swpc']
  ai_watch: ['arxiv', 'huggingface', 'ai-news-rss']
  health: ['who-don', 'cdc', 'outbreak-news']
  sanctions: ['ofac-sdn']
  elections: ['election-calendar']
  shipping: ['yahoo-finance']
  social: ['reddit-public']
  nuclear: ['usgs-nuclear-monitor']
  service_status: ['aws', 'azure', 'gcp', 'cloudflare', 'github']
  geospatial: ['static-datasets (bases, ports, pipelines, nuclear,

## 6. Custom Tool Call
Call any tool by name with arbitrary arguments.

In [23]:
# ---- Customize these ----
TOOL_NAME = "intel_gdelt_search"
ARGUMENTS = {"query": "taiwan strait", "mode": "artlist", "limit": 10}
# -------------------------

result = client.call_tool(TOOL_NAME, ARGUMENTS, timeout=30.0)
display(JSON(result))

<IPython.core.display.JSON object>

## 7. Disconnect

In [ ]:
client.disconnect()